Taller 5 redes neuronales

In [109]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [110]:
import sys
print(sys.executable)


/usr/bin/python3


In [111]:
%cd /content/drive/MyDrive/Riesgos



/content/drive/MyDrive/Riesgos


In [112]:
!ls


label_encoders.joblib  minmax_scaler.joblib  riesgo.xlsx
mejor_modelo.keras     modelo_pca.joblib


In [113]:
import pandas as pd

df = pd.read_excel('riesgo.xlsx')


In [114]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12500 entries, 0 to 12499
Data columns (total 26 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Customer_ID               12500 non-null  object 
 1   Name                      12500 non-null  object 
 2   Age                       12500 non-null  float64
 3   SSN                       12500 non-null  object 
 4   Occupation                12500 non-null  object 
 5   Annual_Income             12500 non-null  float64
 6   Monthly_Inhand_Salary     12500 non-null  float64
 7   Num_Bank_Accounts         12500 non-null  float64
 8   Num_Credit_Card           12500 non-null  float64
 9   Interest_Rate             12500 non-null  int64  
 10  Num_of_Loan               12500 non-null  int64  
 11  Type_of_Loan              11074 non-null  object 
 12  Delay_from_due_date       12500 non-null  float64
 13  Num_of_Delayed_Payment    12500 non-null  float64
 14  Change

In [115]:
df.isnull().sum()


,0
Customer_ID,0
Name,0
Age,0
SSN,0
Occupation,0
Annual_Income,0
Monthly_Inhand_Salary,0
Num_Bank_Accounts,0
Num_Credit_Card,0
Interest_Rate,0


In [116]:
df.drop('Type_of_Loan', axis=1, inplace=True)


In [117]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
Age,12500.0,33.311294,10.760177,14.000000,24.415179,33.000000,41.750000,56.000000
Annual_Income,12500.0,50505.123449,38300.762656,7005.930000,19342.972500,36999.705000,71683.470000,179987.280000
Monthly_Inhand_Salary,12500.0,4198.468568,3187.142979,303.645417,1625.744479,3097.016667,5961.664375,15204.633333
Num_Bank_Accounts,12500.0,5.368828,2.592493,0.000000,3.000000,5.375000,7.000000,10.500000
Num_Credit_Card,12500.0,5.533620,2.066040,0.500000,4.000000,5.000000,7.000000,10.875000
Interest_Rate,12500.0,14.532080,8.741636,1.000000,7.000000,13.000000,20.000000,34.000000
Num_of_Loan,12500.0,3.532880,2.446442,0.000000,2.000000,3.000000,5.000000,9.000000
Delay_from_due_date,12500.0,21.068780,14.772965,-2.000000,9.875000,17.875000,28.000000,63.250000
Num_of_Delayed_Payment,12500.0,13.338642,6.153148,0.000000,9.000000,13.750000,18.175000,26.375000
Changed_Credit_Limit,12500.0,10.465068,6.445141,0.500000,5.493750,9.370000,14.656250,31.115000


In [118]:
df.drop('Name', axis=1, inplace=True)
df.drop('Customer_ID', axis=1, inplace=True)
df.drop('SSN', axis=1, inplace=True)
df.drop('Occupation', axis=1, inplace=True)
df.drop('Payment_Behaviour', axis=1, inplace=True)

In [119]:
for col in df.select_dtypes(include='object').columns:
    print(f"\nFrecuencia de valores en {col}:")
    print(df[col].value_counts())



Frecuencia de valores en Credit_Mix:
Credit_Mix
Standard    5731
Good        3798
Bad         2971
Name: count, dtype: int64

Frecuencia de valores en Payment_of_Min_Amount:
Payment_of_Min_Amount
Yes    7360
No     5031
NM      109
Name: count, dtype: int64


In [120]:
df['Credit_Mix'].nunique()


3

In [121]:
df['Credit_Mix'].unique()


array(['Bad', 'Standard', 'Good'], dtype=object)

In [122]:
df['Credit_Score'].head(10)


,Credit_Score
0,0
1,1
2,0
3,1
4,1
5,2
6,0
7,1
8,2
9,1


In [123]:
df['Credit_Score'].unique()

array([0, 1, 2])

In [124]:
X = df.drop('Credit_Score', axis=1)
y = df['Credit_Score']

In [125]:
X.head()


,Age,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,Num_Credit_Card,Interest_Rate,Num_of_Loan,Delay_from_due_date,Num_of_Delayed_Payment,Changed_Credit_Limit,Num_Credit_Inquiries,Credit_Mix,Outstanding_Debt,Credit_Utilization_Ratio,Credit_History_Age,Payment_of_Min_Amount,Total_EMI_per_month,Amount_invested_monthly,Monthly_Balance
0,17.375,30625.94,2706.161667,6.0,5.0,27,2,62.25,25.000000,1.880000,10.875000,Bad,1562.91,33.477546,10.458333,Yes,42.941090,158.549735,335.375341
1,25.750,52312.68,4250.390000,6.0,5.0,17,4,7.25,17.857143,9.730000,3.000000,Standard,202.68,29.839984,30.714286,Yes,108.366467,146.679378,428.743155
2,18.500,113781.39,9549.782500,1.0,4.0,1,0,13.50,7.375000,10.911429,1.857143,Good,1030.20,34.841449,15.571429,No,0.000000,505.386526,781.229776
3,43.875,58918.47,5208.872500,3.0,3.0,17,3,27.25,14.500000,14.170000,7.000000,Standard,473.14,27.655897,15.541667,Yes,123.434939,311.060914,332.642837
4,43.750,98620.98,7962.415000,3.0,3.0,6,3,12.50,8.428571,1.705000,3.000000,Good,1233.51,31.933940,17.535714,No,228.018084,355.442408,472.781009


In [126]:
from sklearn.preprocessing import LabelEncoder
import joblib

# columnas categóricas
cat_cols = X.select_dtypes(include='object').columns

encoders = {}

for col in cat_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))
    encoders[col] = le

# guardar los encoders
joblib.dump(encoders, 'label_encoders.joblib')


['label_encoders.joblib']

In [127]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12500 entries, 0 to 12499
Data columns (total 19 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Age                       12500 non-null  float64
 1   Annual_Income             12500 non-null  float64
 2   Monthly_Inhand_Salary     12500 non-null  float64
 3   Num_Bank_Accounts         12500 non-null  float64
 4   Num_Credit_Card           12500 non-null  float64
 5   Interest_Rate             12500 non-null  int64  
 6   Num_of_Loan               12500 non-null  int64  
 7   Delay_from_due_date       12500 non-null  float64
 8   Num_of_Delayed_Payment    12500 non-null  float64
 9   Changed_Credit_Limit      12500 non-null  float64
 10  Num_Credit_Inquiries      12500 non-null  float64
 11  Credit_Mix                12500 non-null  int64  
 12  Outstanding_Debt          12500 non-null  float64
 13  Credit_Utilization_Ratio  12500 non-null  float64
 14  Credit

In [128]:
for col, le in encoders.items():
    print(f"{col}: {le.classes_}")


Credit_Mix: ['Bad' 'Good' 'Standard']
Payment_of_Min_Amount: ['NM' 'No' 'Yes']


In [129]:
from sklearn.feature_selection import SelectKBest, f_classif

selector = SelectKBest(score_func=f_classif, k=8)

X_Selected = selector.fit_transform(X, y)


In [130]:
selected_features = X.columns[selector.get_support()]
print(selected_features)


Index(['Num_Bank_Accounts', 'Num_Credit_Card', 'Interest_Rate',
       'Delay_from_due_date', 'Num_of_Delayed_Payment', 'Num_Credit_Inquiries',
       'Outstanding_Debt', 'Payment_of_Min_Amount'],
      dtype='object')


In [131]:
encoded_selected = [col for col in selected_features if col in encoders]

print(encoded_selected)


['Payment_of_Min_Amount']


In [132]:
X[selected_features].agg(['min','max'])


,Num_Bank_Accounts,Num_Credit_Card,Interest_Rate,Delay_from_due_date,Num_of_Delayed_Payment,Num_Credit_Inquiries,Outstanding_Debt,Payment_of_Min_Amount
min,0.0,0.500,1,-2.00,0.000,0.000,0.23,0
max,10.5,10.875,34,63.25,26.375,16.375,4998.07,2


In [133]:
df['Credit_Score'].unique()

array([0, 1, 2])

In [ ]:
from tensorflow.keras.utils import to_categorical

y = pd.Categorical(df['Credit_Score']).codes
y = to_categorical(y, num_classes=3)


In [ ]:
from sklearn.decomposition import PCA
import pandas as pd

pca = PCA(n_components=5)

X_pca = pca.fit_transform(X_Selected)


In [ ]:
X_pca = pd.DataFrame(X_pca, columns=['PC1','PC2','PC3','PC4','PC5'])
X_pca.head()


In [ ]:
sum(pca.explained_variance_ratio_)



In [ ]:
import joblib

joblib.dump(pca, 'modelo_pca.joblib')


In [ ]:
from sklearn.preprocessing import MinMaxScaler
import joblib

scaler = MinMaxScaler()

X_scaled = scaler.fit_transform(X_pca)

# guardar el scaler
joblib.dump(scaler, 'minmax_scaler.joblib')


In [ ]:
X_scaled[:10]


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.15, random_state=42, stratify=y
)


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input

model = Sequential()

# capa de entrada (X_train tiene 5 features)
model.add(Input(shape=(5,)))

# capas ocultas
model.add(Dense(64, activation='relu'))
model.add(Dense(32, activation='relu'))
model.add(Dense(16, activation='relu'))
model.add(Dense(8, activation='relu'))

# capa de salida (3 clases)
model.add(Dense(3, activation='softmax'))

model.summary()


In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.optimizers import Adam
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)


checkpoint = ModelCheckpoint(
    'mejor_modelo.keras',
    monitor='val_accuracy',
    save_best_only=True,
    mode='max',
    verbose=1
)

history = model.fit(
    X_train,
    y_train,
    epochs=100,

    batch_size=32,
    validation_split=0.15,
    verbose=1,
    callbacks=[checkpoint]
)


In [ ]:
import tensorflow as tf
import keras

print("TensorFlow:", tf.__version__)
print("Keras:", keras.__version__)
print("Python:", sys.version)

In [ ]:
from tensorflow.keras.models import load_model

modelo = load_model('mejor_modelo.keras')

In [ ]:
import matplotlib.pyplot as plt

# accuracy
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('Accuracy del modelo')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'])
plt.show()

# loss
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('Loss del modelo')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'])
plt.show()
